In [12]:
# https://github.com/WillKoehrsen/wikipedia-data-science/blob/master/notebooks/Book%20Recommendation%20System.ipynb

import json
from itertools import chain
from collections import Counter, OrderedDict


In [7]:
books = []
with open('/data/jinkilee.github.io/found_books_filtered.ndjson', 'r') as fin:
    # Append each line to the books
    books = [json.loads(l) for l in fin]

In [8]:
# Remove non-book articles
books_with_wikipedia = [book for book in books if 'Wikipedia:' in book[0]]
books = [book for book in books if 'Wikipedia:' not in book[0]]
print('Found {} books.'.format(len(books)))

Found 37020 books.


In [10]:
[book[0] for book in books_with_wikipedia][:5]

['Wikipedia:Wikipedia Signpost/2014-06-25/Recent research',
 'Wikipedia:New pages patrol/Unpatrolled articles/December 2010',
 'Wikipedia:Templates for discussion/Log/2012 September 23',
 'Wikipedia:Articles for creation/Redirects/2012-10',
 'Wikipedia:Templates for discussion/Log/2012 October 4']

In [11]:
book_index = {book[0]: idx for idx, book in enumerate(books)}
index_book = {idx: book for book, idx in book_index.items()}

book_index['Anna Karenina']
index_book[22494]

'Anna Karenina'

In [14]:
wikilinks = list(chain(*[book[2] for book in books]))
print('There are {} unique wikilinks.'.format(len(set(wikilinks))))

There are 311276 unique wikilinks.


In [17]:
n = 21
books[n][2]

['Emmanuel Carrère',
 'biographical novel',
 'Emmanuel Carrère',
 'Eduard Limonov',
 'Prix de la langue française',
 'Prix Renaudot',
 '2011 in literature',
 'Contemporary French literature',
 'Category:2011 French novels',
 'Category:French biographies',
 'Category:Biographical novels',
 'Category:National Bolshevism']

In [19]:
wikilinks_other_books = [link for link in wikilinks if link in book_index.keys()]
print('There are {} unique wikilinks to other books.'.format(len(set(wikilinks_other_books))))

There are 17032 unique wikilinks to other books.


In [22]:
def count_items(l):
    """Return ordered dictionary of counts of objects in `l`"""
    
    # Create a counter object
    counts = Counter(l)
    
    # Sort by highest count first and place in ordered dictionary
    counts = sorted(counts.items(), key = lambda x: x[1], reverse = True)
    counts = OrderedDict(counts)
    
    return counts

In [23]:
# Find set of wikilinks for each book and convert to a flattened list
unique_wikilinks = list(chain(*[list(set(book[2])) for book in books]))

wikilink_counts = count_items(unique_wikilinks)
list(wikilink_counts.items())[:10]

[('Hardcover', 7489),
 ('Paperback', 7311),
 ('Wikipedia:WikiProject Books', 6043),
 ('Wikipedia:WikiProject Novels', 6015),
 ('English language', 4185),
 ('United States', 3060),
 ('Science fiction', 3030),
 ('The New York Times', 2727),
 ('science fiction', 2502),
 ('novel', 1979)]

In [31]:
wikilinks = [link.lower() for link in unique_wikilinks]
print('There are {} unique wikilinks.'.format(len(set(wikilinks))))

wikilink_counts = count_items(wikilinks)
list(wikilink_counts.items())[:10]

There are 297624 unique wikilinks.


[('paperback', 8740),
 ('hardcover', 8648),
 ('wikipedia:wikiproject books', 6043),
 ('wikipedia:wikiproject novels', 6016),
 ('science fiction', 5665),
 ('english language', 4248),
 ('united states', 3063),
 ('novel', 2983),
 ('the new york times', 2742),
 ('fantasy', 2003)]

In [32]:
to_remove = ['hardcover', 'paperback', 'hardback', 'e-book', 'wikipedia:wikiproject books', 'wikipedia:wikiproject novels']
for t in to_remove:
    wikilinks.remove(t)
    _ = wikilink_counts.pop(t)

In [33]:
# Limit to greater than 3 links
links = [t[0] for t in wikilink_counts.items() if t[1] >= 4]
print(len(links))

41758


In [34]:
# Find set of book wikilinks for each book
unique_wikilinks_books = list(chain(*[list(set(link for link in book[2] if link in book_index.keys())) for book in books]))

# Count the number of books linked to by other books
wikilink_book_counts = count_items(unique_wikilinks_books)
list(wikilink_book_counts.items())[:10]

[('The Encyclopedia of Science Fiction', 127),
 ('The Discontinuity Guide', 104),
 ('The Encyclopedia of Fantasy', 63),
 ('Dracula', 55),
 ('Encyclopædia Britannica', 51),
 ('Nineteen Eighty-Four', 51),
 ('The Wonderful Wizard of Oz', 49),
 ('Don Quixote', 49),
 ("Alice's Adventures in Wonderland", 47),
 ('Jane Eyre', 39)]

In [35]:
for book in books:
    if 'The New York Times' in book[2] and 'New York Times' in book[2]:
        print(book[0], book[2])
        break

The Big Picture: Who Killed Hollywood? and Other Essays ['Wikipedia:WikiProject Novels', 'Wikipedia:WikiProject Books', 'William Goldman', 'United States', 'English language', 'William Goldman', 'Michael Sragow', 'Good Will Hunting', 'Robin Williams', 'Matt Damon', 'The New York Times', 'The New York Times Company', 'New York Times', 'Category:Cinema of the United States', 'Category:Film production', 'Category:2000 books', 'Category:Books about films', 'Category:Books by William Goldman', 'Category:Show business memoirs']


In [38]:
link_index = {link: idx for idx, link in enumerate(links)}
index_link = {idx: link for link, idx in link_index.items()}

link_index['the economist']
index_link[300]
print('There are {} wikilinks that will be used.'.format(len(link_index)))

There are 41758 wikilinks that will be used.


# supervised learning

In [42]:
pairs = []

# Iterate through each book
for i, book in enumerate(books):
    if i%100 == 0:
        print('%dth iteration is finished' % i)
    # Iterate through the links in the book
    pairs.extend((book_index[book[0]], link_index[link.lower()]) for link in book[2] if link.lower() in links)
    
len(pairs), len(links), len(books)
pairs[5000]

0th iteration is finished
100th iteration is finished
200th iteration is finished
300th iteration is finished
400th iteration is finished
500th iteration is finished
600th iteration is finished
700th iteration is finished
800th iteration is finished
900th iteration is finished
1000th iteration is finished
1100th iteration is finished
1200th iteration is finished
1300th iteration is finished
1400th iteration is finished
1500th iteration is finished
1600th iteration is finished
1700th iteration is finished
1800th iteration is finished
1900th iteration is finished
2000th iteration is finished
2100th iteration is finished
2200th iteration is finished
2300th iteration is finished
2400th iteration is finished
2500th iteration is finished
2600th iteration is finished
2700th iteration is finished
2800th iteration is finished
2900th iteration is finished
3000th iteration is finished
3100th iteration is finished
3200th iteration is finished
3300th iteration is finished
3400th iteration is finish

27700th iteration is finished
27800th iteration is finished
27900th iteration is finished
28000th iteration is finished
28100th iteration is finished
28200th iteration is finished
28300th iteration is finished
28400th iteration is finished
28500th iteration is finished
28600th iteration is finished
28700th iteration is finished
28800th iteration is finished
28900th iteration is finished
29000th iteration is finished
29100th iteration is finished
29200th iteration is finished
29300th iteration is finished
29400th iteration is finished
29500th iteration is finished
29600th iteration is finished
29700th iteration is finished
29800th iteration is finished
29900th iteration is finished
30000th iteration is finished
30100th iteration is finished
30200th iteration is finished
30300th iteration is finished
30400th iteration is finished
30500th iteration is finished
30600th iteration is finished
30700th iteration is finished
30800th iteration is finished
30900th iteration is finished
31000th it

(321, 232)

In [44]:
pairs_set = set(pairs)

In [47]:
x = Counter(pairs)
sorted(x.items(), key = lambda x: x[1], reverse = True)[:5]

[((13337, 26946), 85),
 ((31899, 65), 77),
 ((25899, 9233), 61),
 ((1851, 2648), 57),
 ((25899, 27804), 53)]

In [48]:
import numpy as np
import random
random.seed(100)

def generate_batch(pairs, n_positive = 50, negative_ratio = 1.0, classification = False):
    """Generate batches of samples for training"""
    batch_size = n_positive * (1 + negative_ratio)
    batch = np.zeros((batch_size, 3))
    
    # Adjust label based on task
    if classification:
        neg_label = 0
    else:
        neg_label = -1
    
    # This creates a generator
    while True:
        # randomly choose positive examples
        for idx, (book_id, link_id) in enumerate(random.sample(pairs, n_positive)):
            batch[idx, :] = (book_id, link_id, 1)

        # Increment idx by 1
        idx += 1
        
        # Add negative examples until reach batch size
        while idx < batch_size:
            
            # random selection
            random_book = random.randrange(len(books))
            random_link = random.randrange(len(links))
            
            # Check to make sure this is not a positive example
            if (random_book, random_link) not in pairs_set:
                
                # Add to batch and increment index
                batch[idx, :] = (random_book, random_link, neg_label)
                idx += 1
                
        # Make sure to shuffle order
        np.random.shuffle(batch)
        yield {'book': batch[:, 0], 'link': batch[:, 1]}, batch[:, 2]

In [62]:
from keras.layers import Input, Embedding, Dot, Reshape, Dense
from keras.models import Model

Using TensorFlow backend.


In [63]:
def book_embedding_model(embedding_size = 50, classification = False):
    """Model to embed books and wikilinks using the functional API.
       Trained to discern if a link is present in a article"""
    
    # Both inputs are 1-dimensional
    book = Input(name = 'book', shape = [1])
    link = Input(name = 'link', shape = [1])
    
    # Embedding the book (shape will be (None, 1, 50))
    book_embedding = Embedding(name = 'book_embedding',
                               input_dim = len(book_index),
                               output_dim = embedding_size)(book)
    
    # Embedding the link (shape will be (None, 1, 50))
    link_embedding = Embedding(name = 'link_embedding',
                               input_dim = len(link_index),
                               output_dim = embedding_size)(link)
    
    # Merge the layers with a dot product along the second axis (shape will be (None, 1, 1))
    merged = Dot(name = 'dot_product', normalize = True, axes = 2)([book_embedding, link_embedding])
    
    # Reshape to be a single number (shape will be (None, 1))
    merged = Reshape(target_shape = [1])(merged)
    
    # If classifcation, add extra layer and loss function is binary cross entropy
    if classification:
        merged = Dense(1, activation = 'sigmoid')(merged)
        model = Model(inputs = [book, link], outputs = merged)
        model.compile(optimizer = 'Adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
    
    # Otherwise loss function is mean squared error
    else:
        model = Model(inputs = [book, link], outputs = merged)
        model.compile(optimizer = 'Adam', loss = 'mse')
    
    return model


In [64]:
# Instantiate model and show parameters
model = book_embedding_model()
model.summary()

__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
book (InputLayer)               (None, 1)            0                                            
__________________________________________________________________________________________________
link (InputLayer)               (None, 1)            0                                            
__________________________________________________________________________________________________
book_embedding (Embedding)      (None, 1, 50)        1851000     book[0][0]                       
__________________________________________________________________________________________________
link_embedding (Embedding)      (None, 1, 50)        2087900     link[0][0]                       
__________________________________________________________________________________________________
dot_produc

In [65]:
n_positive = 1024
gen = generate_batch(pairs, n_positive, negative_ratio = 2)

In [66]:
# Train
h = model.fit_generator(gen, epochs = 15, 
                        steps_per_epoch = len(pairs) // n_positive,
                        verbose = 2)

Epoch 1/15
 - 9s - loss: 0.9638
Epoch 2/15
 - 8s - loss: 0.7868
Epoch 3/15
 - 8s - loss: 0.5521
Epoch 4/15
 - 8s - loss: 0.5007
Epoch 5/15
 - 8s - loss: 0.4821
Epoch 6/15
 - 8s - loss: 0.4701
Epoch 7/15
 - 8s - loss: 0.4661
Epoch 8/15
 - 8s - loss: 0.4573
Epoch 9/15
 - 8s - loss: 0.4536
Epoch 10/15
 - 8s - loss: 0.4587
Epoch 11/15
 - 8s - loss: 0.4549
Epoch 12/15
 - 8s - loss: 0.4468
Epoch 13/15
 - 8s - loss: 0.4446
Epoch 14/15
 - 8s - loss: 0.4438
Epoch 15/15
 - 8s - loss: 0.4452


In [67]:
model.save('/data/jinkilee.github.io/book_recommendation.h5')

In [83]:
import tensorflow as tf

embedding_size = 50

# Both inputs are 1-dimensional
book = Input(name = 'book', shape = [1])
link = Input(name = 'link', shape = [1])

# Embedding the book (shape will be (None, 1, 50))
book_embedding = Embedding(name = 'book_embedding',
                           input_dim = len(book_index),
                           output_dim = embedding_size)(book)

# Embedding the link (shape will be (None, 1, 50))
link_embedding = Embedding(name = 'link_embedding',
                           input_dim = len(link_index),
                           output_dim = embedding_size)(link)

# Merge the layers with a dot product along the second axis (shape will be (None, 1, 1))
# Merge the layers with a dot product along the second axis (shape will be (None, 1, 1))
book_embedding = tf.placeholder(tf.float32, shape=(None, 1, 50))
link_embedding = tf.placeholder(tf.float32, shape=(None, 1, 50))
merged = Dot(name = 'dot_product', normalize = True, axes = 2)([book_embedding, link_embedding])
merged
# Reshape to be a single number (shape will be (None, 1))
# merged = Reshape(target_shape = [1])(merged)

<tf.Tensor 'dot_product_4/MatMul:0' shape=(?, 1, 1) dtype=float32>

In [100]:
#book_embedding.shape, link_embedding.shape, merged.shape
with tf.Session() as sess:
    zeros = np.zeros((10, 1, 50)) + np.ones((10, 1, 50)) + np.ones((10, 1, 50)) + np.ones((10, 1, 50)) + np.ones((10, 1, 50))
    ones = np.ones((10, 1, 50)) + np.ones((10, 1, 50))
    feed_dict={book_embedding:zeros, link_embedding:ones}
    mrg = sess.run(merged, feed_dict=feed_dict)
    mrg

In [105]:
# Extract embeddings
book_layer = model.get_layer('book_embedding')
book_weights = book_layer.get_weights()[0]
book_weights.shape

(37020, 50)

In [118]:
book_weights = book_weights / np.linalg.norm(book_weights, axis = 1).reshape((-1, 1))
book_weights[0][:10]
np.sum(np.square(book_weights[0]))

1.0

In [119]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('fivethirtyeight')
plt.rcParams['font.size'] = 15

def find_similar(name, weights, index_name = 'book', n = 10, least = False, return_dist = False, plot = False):
    """Find n most similar items (or least) to name based on embeddings. Option to also plot the results"""
    
    # Select index and reverse index
    if index_name == 'book':
        index = book_index
        rindex = index_book
    elif index_name == 'page':
        index = link_index
        rindex = index_link
    
    # Check to make sure `name` is in index
    try:
        # Calculate dot product between book and all others
        dists = np.dot(weights, weights[index[name]])
    except KeyError:
        print('{} Not Found.'.format(name))
        return
    
    # Sort distance indexes from smallest to largest
    sorted_dists = np.argsort(dists)
    
    # Plot results if specified
    if plot:
        
        # Find furthest and closest items
        furthest = sorted_dists[:(n // 2)]
        closest = sorted_dists[-n-1: len(dists) - 1]
        items = [rindex[c] for c in furthest]
        items.extend(rindex[c] for c in closest)
        
        # Find furthest and closets distances
        distances = [dists[c] for c in furthest]
        distances.extend(dists[c] for c in closest)
        
        colors = ['r' for _ in range(n //2)]
        colors.extend('g' for _ in range(n))
        
        data = pd.DataFrame({'distance': distances}, index = items)
        
        # Horizontal bar chart
        data['distance'].plot.barh(color = colors, figsize = (10, 8),
                                   edgecolor = 'k', linewidth = 2)
        plt.xlabel('Cosine Similarity');
        plt.axvline(x = 0, color = 'k');
        
        # Formatting for italicized title
        name_str = '{}s Most and Least Similar to'.format(index_name.capitalize())
        for word in name.split():
            # Title uses latex for italize
            name_str += ' $\it{' + word + '}$'
        plt.title(name_str, x = 0.2, size = 28, y = 1.05)
        
        return None
    
    # If specified, find the least similar
    if least:
        # Take the first n from sorted distances
        closest = sorted_dists[:n]
         
        print('{}s furthest from {}.\n'.format(index_name.capitalize(), name))
        
    # Otherwise find the most similar
    else:
        # Take the last n sorted distances
        closest = sorted_dists[-n:]
        
        # Need distances later on
        if return_dist:
            return dists, closest
        
        
        print('{}s closest to {}.\n'.format(index_name.capitalize(), name))
        
    # Need distances later on
    if return_dist:
        return dists, closest
    
    
    # Print formatting
    max_width = max([len(rindex[c]) for c in closest])
    
    # Print the most similar and distances
    for c in reversed(closest):
        print(index_name.capitalize(), rindex[c], max_width+2, dists[c])

In [120]:
find_similar('War and Peace', book_weights)

Books closest to War and Peace.

Book War and Peace 28 1.0
Book Anna Karenina 28 0.909616
Book Demons (Dostoevsky novel) 28 0.906105
Book The Master and Margarita 28 0.873261
Book The Good Soldier Švejk 28 0.854039
Book Dead Souls 28 0.849149
Book Buddenbrooks 28 0.840407
Book Under Western Eyes (novel) 28 0.835929
Book Eugene Onegin 28 0.832963
Book Doctor Zhivago (novel) 28 0.827927
